## 1) Model Setup


In [1]:
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, cross_val_score, RandomizedSearchCV, cross_val_predict
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree
from sklearn.ensemble import BaggingClassifier, BaggingRegressor, RandomForestRegressor, RandomForestClassifier, StackingClassifier, GradientBoostingClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, mean_absolute_error, r2_score, recall_score, roc_curve, roc_auc_score
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from catboost import CatBoostClassifier
import optuna
from optuna.samplers import TPESampler
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

In [2]:
sample_submission_data = pd.read_csv("sample_submission.csv")
pub_priv_data = pd.read_csv("public_private_X.csv")
train_X_data = pd.read_csv("train_X.csv")
train_y_data = pd.read_csv("train_y.csv")
train_data = pd.merge(train_X_data, train_y_data)

# Fill missing values with median
cols_with_missing_data = ["AVERAGE_DAILY_DEMAND_CASES", "AVERAGE_VENDOR_ORDER_CYCLE_DAYS", "AVERAGE_ORDER_CYCLE_DAYS", "AVERAGE_ORDER_CYCLE_CASES"]

for col in cols_with_missing_data:
    train_data.loc[:, col] = train_data[col].fillna(train_data[col].median())
    pub_priv_data.loc[:, col] = pub_priv_data[col].fillna(pub_priv_data[col].median())

# LOG VARIABLES
# First fix rows with zero and fill with median (will throw an error if not)
train_data.loc[train_data["GIVEN_TIME_TO_LEAD_TIME_RATIO"] <= 0, "GIVEN_TIME_TO_LEAD_TIME_RATIO"] = np.nan
train_data["GIVEN_TIME_TO_LEAD_TIME_RATIO"] = train_data["GIVEN_TIME_TO_LEAD_TIME_RATIO"].fillna(train_data["GIVEN_TIME_TO_LEAD_TIME_RATIO"].median())
pub_priv_data.loc[pub_priv_data["GIVEN_TIME_TO_LEAD_TIME_RATIO"] <= 0, "GIVEN_TIME_TO_LEAD_TIME_RATIO"] = np.nan
pub_priv_data["GIVEN_TIME_TO_LEAD_TIME_RATIO"] = pub_priv_data["GIVEN_TIME_TO_LEAD_TIME_RATIO"].fillna(pub_priv_data["GIVEN_TIME_TO_LEAD_TIME_RATIO"].median())

train_data.loc[train_data["LEAD_TIME_TO_DISTANCE_RATIO"] <= 0, "LEAD_TIME_TO_DISTANCE_RATIO"] = np.nan
train_data["LEAD_TIME_TO_DISTANCE_RATIO"] = train_data["LEAD_TIME_TO_DISTANCE_RATIO"].fillna(train_data["LEAD_TIME_TO_DISTANCE_RATIO"].median())
pub_priv_data.loc[pub_priv_data["LEAD_TIME_TO_DISTANCE_RATIO"] <= 0, "LEAD_TIME_TO_DISTANCE_RATIO"] = np.nan
pub_priv_data["LEAD_TIME_TO_DISTANCE_RATIO"] = pub_priv_data["LEAD_TIME_TO_DISTANCE_RATIO"].fillna(pub_priv_data["LEAD_TIME_TO_DISTANCE_RATIO"].median())

# Create log variables for both training and test sets
train_data["log_time_to_lead"] = np.log(train_data["GIVEN_TIME_TO_LEAD_TIME_RATIO"] + 1e-10)
train_data["log_quantity_market"] = np.log(train_data["AVERAGE_PRODUCT_ORDER_QUANTITY_MARKET"] + 1e-10)
train_data["log_time_dist"] = np.log(train_data["LEAD_TIME_TO_DISTANCE_RATIO"] + 1e-10)
train_data["log_ship_vendor"] = np.log(train_data["SHIP_FROM_VENDOR"] + 1e-10)

pub_priv_data["log_time_to_lead"] = np.log(pub_priv_data["GIVEN_TIME_TO_LEAD_TIME_RATIO"] + 1e-10)
pub_priv_data["log_quantity_market"] = np.log(pub_priv_data["AVERAGE_PRODUCT_ORDER_QUANTITY_MARKET"] + 1e-10)
pub_priv_data["log_time_dist"] = np.log(pub_priv_data["LEAD_TIME_TO_DISTANCE_RATIO"] + 1e-10)
pub_priv_data["log_ship_vendor"] = np.log(pub_priv_data["SHIP_FROM_VENDOR"] + 1e-10)

# Bin purchase order due date variable for both train and test
train_data['PURCHASE_ORDER_DUE_DATE'] = pd.to_datetime(train_data['PURCHASE_ORDER_DUE_DATE'])
train_data['week_of_year'] = train_data['PURCHASE_ORDER_DUE_DATE'].dt.isocalendar().week
train_data['week_of_year'] = train_data['week_of_year'].astype(int)

pub_priv_data['PURCHASE_ORDER_DUE_DATE'] = pd.to_datetime(pub_priv_data['PURCHASE_ORDER_DUE_DATE'])
pub_priv_data['week_of_year'] = pub_priv_data['PURCHASE_ORDER_DUE_DATE'].dt.isocalendar().week
pub_priv_data['week_of_year'] = pub_priv_data['week_of_year'].astype(int)

# Bin days between order and due date variable for both train and test
day_binned = pd.qcut(train_data['DAYS_BETWEEN_ORDER_AND_DUE_DATE'], q=15, retbins=True, duplicates='drop')
bins = day_binned[1]

train_data['order_to_due_binned'] = pd.cut(train_data['DAYS_BETWEEN_ORDER_AND_DUE_DATE'], bins=bins, include_lowest=True, labels=False)
pub_priv_data['order_to_due_binned'] = pd.cut(pub_priv_data['DAYS_BETWEEN_ORDER_AND_DUE_DATE'], bins=bins, include_lowest=True, labels=False)

## 2) Model Training

Building and training of base models.


In [3]:
# define features and target
X_train = train_data[['PRODUCT_CLASSIFICATION', 'PURCHASE_FROM_VENDOR',
                'AVERAGE_DAILY_DEMAND_CASES', 'AVERAGE_VENDOR_ORDER_CYCLE_DAYS',
                'AVERAGE_ORDER_CYCLE_DAYS', 'DISTANCE_IN_MILES',
                'log_time_to_lead', 'log_quantity_market', 'DIVISION_CODE',
                'week_of_year', 'PURCHASING_LEAD_TIME', 'log_time_dist',
                'log_ship_vendor', 'order_to_due_binned', 'DUE_DATE_WEEKDAY',
                'TRANSIT_LEAD_TIME', 'AVERAGE_ORDER_CYCLE_CASES']]
y_train = train_data['ON_TIME_AND_COMPLETE']
X_test_final = pub_priv_data[X_train.columns]

# separate numerical and categorical features
num_features = ['AVERAGE_DAILY_DEMAND_CASES', 'AVERAGE_VENDOR_ORDER_CYCLE_DAYS',
                'AVERAGE_ORDER_CYCLE_DAYS', 'DISTANCE_IN_MILES', 'log_time_to_lead',
                'log_quantity_market', 'log_time_dist', 'PURCHASING_LEAD_TIME',
                'log_ship_vendor', 'AVERAGE_ORDER_CYCLE_CASES']
cat_features = ['PRODUCT_CLASSIFICATION', 'PURCHASE_FROM_VENDOR', 'DIVISION_CODE',
                'order_to_due_binned', 'DUE_DATE_WEEKDAY', 'TRANSIT_LEAD_TIME']

# preprocessing
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_features),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_features)
])

# base learners
base_learners = [
    ('rf', RandomForestClassifier(random_state=42)),
    ('cat', CatBoostClassifier(random_state=42, verbose=0)),
    ('xgb', XGBClassifier(eval_metric='logloss', random_state=42))
]

# stacking model
stacking_model = Pipeline([
    ('preprocessor', preprocessor),
    ('stacking', StackingClassifier(
        estimators=base_learners,
        final_estimator=LogisticRegression(max_iter=1000, random_state=42),
        cv=5
    ))
])

# fit stacking model
stacking_model.fit(X_train, y_train)


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['AVERAGE_DAILY_DEMAND_CASES',
                                                   'AVERAGE_VENDOR_ORDER_CYCLE_DAYS',
                                                   'AVERAGE_ORDER_CYCLE_DAYS',
                                                   'DISTANCE_IN_MILES',
                                                   'log_time_to_lead',
                                                   'log_quantity_market',
                                                   'log_time_dist',
                                                   'PURCHASING_LEAD_TIME',
                                                   'log_ship_vendor',
                                                   'AVERAGE_ORDER_CYCLE_CASES']),
                                                 ('cat',
                                                  OneHotEncoder(ha...
                                                               interaction_constraints=None,
                                                               learning_rate=None,
                                                               max_bin=None,
                                                               max_cat_threshold=None,
                                                               max_cat_to_onehot=None,
                                                               max_delta_step=None,
                                                               max_depth=None,
                                                               max_leaves=None,
                                                               min_child_weight=None,
                                                               missing=nan,
                                                               monotone_constraints=None,
                                                               multi_strategy=None,
                                                               n_estimators=None,
                                                               n_jobs=None,
                                                               num_parallel_tree=None, ...))],
                                    final_estimator=LogisticRegression(max_iter=1000,
                                                                       random_state=42)))])

## 3) Hyperparameter Tuning



In [4]:
def rf_objective(trial):
    rf = RandomForestClassifier(
        n_estimators=trial.suggest_int('n_estimators', 50, 300),
        max_depth=trial.suggest_int('max_depth', 5, 20),
        min_samples_split=trial.suggest_int('min_samples_split', 2, 15),
        min_samples_leaf=trial.suggest_int('min_samples_leaf', 1, 10),
        random_state=42
    )
    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', rf)
    ])
    score = cross_val_score(pipe, X_train, y_train, cv=3, scoring='roc_auc').mean()
    return score

rf_study = optuna.create_study(direction='maximize')
rf_study.optimize(rf_objective, n_trials=15)
print("Best RF Params:", rf_study.best_params)


[I 2025-06-02 16:52:49,997] A new study created in memory with name: no-name-82cd4dd6-b8cb-4b86-8fc0-6f456fca175a


[I 2025-06-02 16:52:59,829] Trial 0 finished with value: 0.8876008859766146 and parameters: {'n_estimators': 270, 'max_depth': 20, 'min_samples_split': 8, 'min_samples_leaf': 5}. Best is trial 0 with value: 0.8876008859766146.
[I 2025-06-02 16:53:02,497] Trial 1 finished with value: 0.8850687052353869 and parameters: {'n_estimators': 89, 'max_depth': 16, 'min_samples_split': 9, 'min_samples_leaf': 4}. Best is trial 0 with value: 0.8876008859766146.
[I 2025-06-02 16:53:05,875] Trial 2 finished with value: 0.8302372623205029 and parameters: {'n_estimators': 175, 'max_depth': 5, 'min_samples_split': 13, 'min_samples_leaf': 4}. Best is trial 0 with value: 0.8876008859766146.
[I 2025-06-02 16:53:11,761] Trial 3 finished with value: 0.8818371310574955 and parameters: {'n_estimators': 188, 'max_depth': 16, 'min_samples_split': 13, 'min_samples_leaf': 5}. Best is trial 0 with value: 0.8876008859766146.
[I 2025-06-02 16:53:14,541] Trial 4 finished with value: 0.8758271135015816 and parameters: 

Best RF Params: {'n_estimators': 179, 'max_depth': 20, 'min_samples_split': 14, 'min_samples_leaf': 3}


In [5]:
def xgb_objective(trial):
    xgb = XGBClassifier(
        n_estimators=trial.suggest_int('n_estimators', 50, 300),
        max_depth=trial.suggest_int('max_depth', 3, 10),
        learning_rate=trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        subsample=trial.suggest_float('subsample', 0.5, 1.0),
        colsample_bytree=trial.suggest_float('colsample_bytree', 0.5, 1.0),
        eval_metric='logloss',
        random_state=42
    )
    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', xgb)
    ])
    score = cross_val_score(pipe, X_train, y_train, cv=3, scoring='roc_auc').mean()
    return score

xgb_study = optuna.create_study(direction='maximize')
xgb_study.optimize(xgb_objective, n_trials=15)
print("Best XGBoost Params:", xgb_study.best_params)

[I 2025-06-02 16:54:08,930] A new study created in memory with name: no-name-30902295-51da-4736-92a4-4a26974dc098
[I 2025-06-02 16:54:10,294] Trial 0 finished with value: 0.8775321491541179 and parameters: {'n_estimators': 135, 'max_depth': 5, 'learning_rate': 0.038681839986572805, 'subsample': 0.5060867157367159, 'colsample_bytree': 0.6681834374839437}. Best is trial 0 with value: 0.8775321491541179.
[I 2025-06-02 16:54:12,125] Trial 1 finished with value: 0.8965604991859957 and parameters: {'n_estimators': 119, 'max_depth': 9, 'learning_rate': 0.03320137427282395, 'subsample': 0.5432050004275027, 'colsample_bytree': 0.6504815630824172}. Best is trial 1 with value: 0.8965604991859957.
[I 2025-06-02 16:54:16,351] Trial 2 finished with value: 0.8989036526174053 and parameters: {'n_estimators': 286, 'max_depth': 10, 'learning_rate': 0.040091282236590864, 'subsample': 0.7336036673426363, 'colsample_bytree': 0.9264794498779005}. Best is trial 2 with value: 0.8989036526174053.
[I 2025-06-02

Best XGBoost Params: {'n_estimators': 217, 'max_depth': 10, 'learning_rate': 0.044410093123910234, 'subsample': 0.98142682861338, 'colsample_bytree': 0.8727591380154864}


In [6]:
def cat_objective(trial):
    cat = CatBoostClassifier(
        iterations=trial.suggest_int('iterations', 100, 300),
        depth=trial.suggest_int('depth', 4, 10),
        learning_rate=trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        l2_leaf_reg=trial.suggest_float('l2_leaf_reg', 1, 10),
        verbose=0,
        random_state=42
    )
    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', cat)
    ])
    score = cross_val_score(pipe, X_train, y_train, cv=3, scoring='roc_auc').mean()
    return score

cat_study = optuna.create_study(direction='maximize')
cat_study.optimize(cat_objective, n_trials=15)
print("Best CatBoost Params:", cat_study.best_params)

[I 2025-06-02 16:54:41,664] A new study created in memory with name: no-name-003590ea-e70c-4af6-b2d2-81aabc10333c
[I 2025-06-02 16:54:49,397] Trial 0 finished with value: 0.8627954374725962 and parameters: {'iterations': 170, 'depth': 5, 'learning_rate': 0.020980651288854287, 'l2_leaf_reg': 4.516643396993353}. Best is trial 0 with value: 0.8627954374725962.
[I 2025-06-02 16:54:59,550] Trial 1 finished with value: 0.898706696440772 and parameters: {'iterations': 229, 'depth': 8, 'learning_rate': 0.02761715207874272, 'l2_leaf_reg': 3.1180731076944577}. Best is trial 1 with value: 0.898706696440772.
[I 2025-06-02 16:55:10,009] Trial 2 finished with value: 0.9066331717389439 and parameters: {'iterations': 216, 'depth': 8, 'learning_rate': 0.11935636277250922, 'l2_leaf_reg': 9.445011433370496}. Best is trial 2 with value: 0.9066331717389439.
[I 2025-06-02 16:55:18,678] Trial 3 finished with value: 0.858531383633046 and parameters: {'iterations': 265, 'depth': 5, 'learning_rate': 0.010383416

Best CatBoost Params: {'iterations': 165, 'depth': 10, 'learning_rate': 0.11254680644812866, 'l2_leaf_reg': 9.770902930103713}


## 4) Final Model Training & Prediction



In [7]:
# created tuned models (with best hyperparams)
tuned_rf = RandomForestClassifier(
    n_estimators=140, 
    max_depth=20,
    min_samples_split=11,
    min_samples_leaf=1,
    random_state=42
)

tuned_xgb = XGBClassifier(
    n_estimators=247, 
    max_depth=6, 
    learning_rate=0.12,
    subsample=0.92, 
    colsample_bytree=0.78,
    eval_metric='logloss',
    random_state=42
)

tuned_cat = CatBoostClassifier(
    iterations=261, 
    depth=7, 
    learning_rate=0.13, 
    l2_leaf_reg=2.73,
    verbose=0, 
    random_state=42
)

# new set of base learners using the tuned models
base_learners_tuned = [
    ('rf', tuned_rf),
    ('xgb', tuned_xgb),
    ('cat', tuned_cat)
]

# build stacking model
stacking_model = Pipeline([
    ('preprocessor', preprocessor),
    ('stacking', StackingClassifier(
        estimators=base_learners_tuned,
        final_estimator=LogisticRegression(max_iter=1000, random_state=42),
        cv=5
    ))
])

# fit model
stacking_model.fit(X_train, y_train)

# predict probabilities on train
X_processed = preprocessor.transform(X_train)
y_prob_stack = stacking_model.named_steps['stacking'].predict_proba(X_processed)[:, 1]

# find optimal threshold
fpr, tpr, thresholds = roc_curve(y_train, y_prob_stack)
optimal_idx = np.argmax(tpr - fpr)
optimal_thresh = thresholds[optimal_idx]
print("Optimal Threshold:", optimal_thresh)

# compute train metrics
y_pred_stack = (y_prob_stack > optimal_thresh).astype(int)
print("Stacking Accuracy:", accuracy_score(y_train, y_pred_stack))
print("Stacking ROC AUC:", roc_auc_score(y_train, y_prob_stack))

# cross-validation
cv_score = cross_val_score(stacking_model, X_train, y_train, cv=5, scoring='accuracy')
print("5-Fold CV Accuracy:", cv_score.mean())

# make final test predictions
X_test_processed = preprocessor.transform(X_test_final)
y_test_prob = stacking_model.named_steps['stacking'].predict_proba(X_test_processed)[:, 1]
y_test_pred = (y_test_prob > optimal_thresh).astype(int)

# save predictions
output = pd.DataFrame({'ID': pub_priv_data['ID'], 'Predicted': y_test_pred})
output.to_csv("predictions_stacking.csv", index=False)

Optimal Threshold: 0.4520794239691965
Stacking Accuracy: 0.862734531915936
Stacking ROC AUC: 0.9370600342375113
5-Fold CV Accuracy: 0.8356930056064732


## 5) Justification for Final Model 



I applied stacking with three base models (Random Forest, XGBoost, and Catboost). I did not want my model to be too computationally expensive, as I did not have entensive time to devote to this. Therefore, I selected three models that I thought would yield me good results while being flexible for hyperparameter tuning. I played around with adding in other models, such as KNN or LightGMB, and found that this combination gave me the best results. Additionally, I tuned each of these models using Optuna with hyperparamter grids. 

## 7) Key Takeaways and Reflection

My main takeaway from this prediction problem is that I should let my results and the data guide me rather than solely focusing on theory. For instance, sometimes I would try something that I assumed would yield better results (such relying on CatBoost's native categorical support rather than one hot encoding), but it would instead perform much worse. Additionally, I was truly able to see the trade off been performance and being computationally expensive. If I had more time or a more powerful computer, I would love to have let my models run longer. When I had the time to (and access to a computer charger because it drained my battery), I noticed my models performed a lot better. Furthermore, I wish I had devised a better method to understand the private leaderboard. While I tried various things to improve generalization and decrease overfitting, there were so many more things I wanted to try but didn't because I wouldn't know if my models were fully performing better or worse. Perhaps if I had checked the csv file and compared the predictions I could have worked on this. 